# Gold — reusable profiles and variability

Derive median nominal pace by synthetic participant and deviations around each session/participant median. Gold profiles remain separate from downstream scenario populations: a simulated 100-driver field never increases the input sample size.


In [ ]:
from pathlib import Path
import json, os, sys
root_hint = os.environ.get("AIDP_PROJECT_ROOT")
if root_hint:
    root = Path(root_hint).resolve()
else:
    root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "race_demo/pipeline.py").is_file()), None)
if root is None or not (root / "race_demo/pipeline.py").is_file():
    raise RuntimeError("Set AIDP_PROJECT_ROOT to this repository checkout.")
sys.path.insert(0, str(root))
from race_demo.pipeline import generate_bronze, build_silver, build_gold, serving_document, write_once, digest
from race_demo.workbench import publish_table, table_name
execution = os.environ.get("RACE_DEMO_EXECUTION", "LOCAL_DEMO")
if execution not in {"LOCAL_DEMO", "AIDP_WORKBENCH"}:
    raise ValueError("Unknown execution mode")
if execution == "AIDP_WORKBENCH":
    if "spark" not in globals():
        raise RuntimeError("Use an AIDP notebook runtime with Spark/Delta configured.")
    table_name("bronze")  # Validate the dedicated demo namespace before writes.
    if not os.environ.get("RACE_DEMO_OUTPUT"):
        raise RuntimeError("Set RACE_DEMO_OUTPUT to a new writable demo volume directory.")
output = Path(os.environ.get("RACE_DEMO_OUTPUT", str(root / "build/notebooks")))
print("Execution:", execution, "| Evidence: SYNTHETIC_DEMO")


In [ ]:
silver = ([r.asDict() for r in spark.table(table_name("silver")).collect()]
          if execution == "AIDP_WORKBENCH" else json.loads((output / "silver/laps.json").read_text()))
gold = build_gold(silver)
write_once(output / "gold/profiles.json", gold)
if execution == "AIDP_WORKBENCH":
    print(publish_table(spark, "gold_profiles", gold["profiles"]))
    print(publish_table(spark, "gold_deviations", [{"sample_index": i, "deviation_seconds": d, "evidence_type": "SYNTHETIC_DEMO"} for i, d in enumerate(gold["deviations"])]))
print({"profiles": len(gold["profiles"]), "deviations": len(gold["deviations"]), "sha256": digest(gold)})
